# 04 – Kontinuální vlnková transformace (CWT): Jemné škály a Morlet s $\omega_0 = 6.0$

### Fyzikální model a klíčová doporučení:
1. **Jemnější škály pro vysoké časové rozlišení náběhu skoku (škály 0.5 až 16.0):**
   - Zmenšením minimální škály na 0.5 dosahujeme sub-sekundové časové lokalizace strmých teplotních a mechanických náběhových hran.
2. **Morletova vlnka s parametrem $\omega_0 = 6.0$ (`cmor1.5-6.0`):**
   - Parametr centrální frekvence $\omega_0 = 6.0$ splňuje striktní podmínku admissibility (zamezuje průsaku záporných frekvencí) a poskytuje ideální kompromis mezi časovým a frekvenčním rozlišením pro optovláknové signály.
3. **Příznak `cwt_fine_scale_onset_energy`:**
   - Okamžitá energie na nejjemnějších škálách ($\le 1.5$) reagující výhradně na skokové změny a impulzní události.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from icaros.analysis.runner import load_windows_with_signals, run_single_silver_method
from icaros.db.s3 import read_parquet_from_s3

# 1. Načtení klouzavých oken
df_win, signals, fs = load_windows_with_signals('win128_stride32')
print(f'Načteno {len(df_win)} oken, fs = {fs} Hz')

# 2. Spuštění CWT s jemnými škálami a Morletem omega0=6
res = run_single_silver_method('cwt', df_win, signals, fs, 'win128_stride32')
print('Metriky separability (CWT Morlet omega0=6):')
for k, v in res.items():
    if k != 's3_uri':
        print(f'  {k}: {v}')

# 3. Načtení výsledků
df_cwt = read_parquet_from_s3(res['s3_uri'])
disp_cols = [
    'measurement_code', 'inherited_label', 'cwt_total_energy',
    'cwt_max_transient_peak', 'cwt_fine_scale_onset_energy',
    'cwt_dominant_scale', 'cwt_wavelet_entropy'
]
display(df_cwt[disp_cols].head(15))


In [ ]:
# Diagnostická vizualizace škálogramů a detekce hran
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# (A) Jemný CWT Morlet (omega0=6) škálogram vybraného přechodového okna
trans_idx = df_cwt[df_cwt['is_transition_window'] == True].index
sample_idx = trans_idx[0] if len(trans_idx) > 0 else 0
cwt_map = np.stack(df_cwt['cwt_scalogram_map'].iloc[sample_idx])
im0 = axes[0, 0].imshow(cwt_map, aspect='auto', origin='lower', cmap='magma')
axes[0, 0].set_title(f'CWT Morlet ($\omega_0=6$) škálogram přechodu ({df_cwt["inherited_label"].iloc[sample_idx]})')
axes[0, 0].set_xlabel('Časový krok v okně')
axes[0, 0].set_ylabel('Škála (0.5 .. 16.0)')
plt.colorbar(im0, ax=axes[0, 0], fraction=0.046, pad=0.04)

# (B) Energie na jemných škálách (cwt_fine_scale_onset_energy) pro přechody vs klid
df_cwt.boxplot(column='cwt_fine_scale_onset_energy', by='is_transition_window', ax=axes[0, 1])
axes[0, 1].set_title('Energie na jemných škálách (Detekce ostrého náběhu skoku)')
axes[0, 1].set_xlabel('is_transition_window (False = ustálený stav, True = skok)')
axes[0, 1].set_ylabel('cwt_fine_scale_onset_energy')
axes[0, 1].set_yscale('log')
axes[0, 1].grid(True, linestyle='--', alpha=0.5)

# (C) Vlnková entropie podle tříd jevů
for lbl, grp in df_cwt.groupby('inherited_label'):
    if len(grp) > 5:
        axes[1, 0].scatter(grp['cwt_dominant_scale'], grp['cwt_wavelet_entropy'], s=14, alpha=0.6, label=str(lbl))
axes[1, 0].set_title('Dominantní škála vs. Vlnková entropie')
axes[1, 0].set_xlabel('Dominantní škála vlnky')
axes[1, 0].set_ylabel('Vlnková entropie')
axes[1, 0].grid(True, linestyle='--', alpha=0.5)
axes[1, 0].legend(fontsize=7, loc='best')

# (D) Transientní špička vs. Celková energie vlnky
for lbl, grp in df_cwt.groupby('inherited_label'):
    if len(grp) > 5:
        axes[1, 1].scatter(grp['cwt_total_energy'], grp['cwt_max_transient_peak'], s=14, alpha=0.6, label=str(lbl))
axes[1, 1].set_title('Celková vlnková energie vs. Maximální přechodová špička')
axes[1, 1].set_xlabel('Celková energie CWT')
axes[1, 1].set_ylabel('cwt_max_transient_peak')
axes[1, 1].set_xscale('log')
axes[1, 1].set_yscale('log')
axes[1, 1].grid(True, linestyle='--', alpha=0.5)
axes[1, 1].legend(fontsize=7, loc='best')

plt.tight_layout()
plt.show()
